<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Analyze System Logs </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

** Exercise: Agent Observability and Monitoring**



### Exercise

A **DevOps CI/CD Pipeline Agent** has been running in production for 48 hours.  
It processes deployment requests by checking code quality, running tests, scanning for security vulnerabilities, and triggering deployments.

Something is wrong. The on-call engineer got paged at 3 AM, and you have been handed **the raw agent logs** from the past 48 hours.

Your job: **analyze the logs, find what's broken, and write a system health report.**

---

###  Scenario: DevOps CI/CD Pipeline Agent
The agent uses four tools to process deployment pipelines:

| Tool | What it does |
|------|-------------|
| `run_code_quality_check` | Lints code and checks style violations |
| `execute_test_suite` | Runs unit + integration tests |
| `scan_security_vulnerabilities` | Scans dependencies for CVEs |
| `trigger_deployment` | Deploys to staging or production |

The agent ran **50 pipeline requests** over 48 hours. The logs have been pre-generated for you.

### Exercise Tasks


**Task 1** Explore the log structure and compute basic statistics

**Task 2** Identify failure patterns across runs and tools

**Task 3** Detect performance anomalies (latency outliers, slow steps)

**Task 4** Build a failure pattern classifier and timeline

**Task 5** Write a structured system health report


## 1. Setup

In [ ]:
!pip install -q langchain langchain-openai langgraph --quiet

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete")

In [ ]:
import json
import random
import statistics
import uuid
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

print("Imports ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 367.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.7

In [ ]:
import time
import uuid
from datetime import datetime
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Imports ready")

LLM ready


## 2. Pre-Generated Log Dataset (Read Only — Do Not Modify)

The cells below simulate 48 hours of agent activity and produce the log dataset you will analyse.  
**Do not modify this section.** Read through it to understand the log structure before you begin the tasks.

### Log Structure
Every **run** is a dict with this shape:

```python
{
  "run_id":       str,        # e.g. "a3f2c1"
  "pipeline_id":  str,        # e.g. "pipe-042"
  "repo":         str,        # e.g. "backend-api"
  "triggered_at": str,        # ISO datetime string
  "status":       str,        # "success" | "failed" | "timeout"
  "total_ms":     float,      # end-to-end duration in milliseconds
  "steps": [                  # ordered list of tool invocations
    {
      "step_no":    int,
      "tool":       str,       # tool name
      "status":     str,       # "ok" | "failed" | "timeout" | "skipped"
      "latency_ms": float,
      "output":     str,       # tool return value / error message
      "error_code": str|None   # e.g. "CVE_SCAN_TIMEOUT", None if ok
    }
  ]
}
```

In [ ]:
#  Log generator — pre-built, do not modify
random.seed(42)

REPOS      = ["backend-api", "frontend-app", "auth-service", "data-pipeline", "ml-model-api"]
ENVS       = ["staging", "production"]
TOOL_NAMES = ["run_code_quality_check", "execute_test_suite",
              "scan_security_vulnerabilities", "trigger_deployment"]

# Failure injection config — this is what you will discover during analysis
FAILURE_CONFIG = {
    "scan_security_vulnerabilities": {"fail_rate": 0.30, "timeout_rate": 0.10,
                                      "slow_rate": 0.25, "slow_multiplier": 3.5},
    "execute_test_suite":            {"fail_rate": 0.12, "timeout_rate": 0.04,
                                      "slow_rate": 0.15, "slow_multiplier": 2.2},
    "run_code_quality_check":        {"fail_rate": 0.05, "timeout_rate": 0.01,
                                      "slow_rate": 0.08, "slow_multiplier": 1.5},
    "trigger_deployment":            {"fail_rate": 0.08, "timeout_rate": 0.02,
                                      "slow_rate": 0.05, "slow_multiplier": 1.3},
}

BASE_LATENCY = {
    "run_code_quality_check":       220,
    "execute_test_suite":           850,
    "scan_security_vulnerabilities":640,
    "trigger_deployment":           310,
}

ERROR_CODES = {
    "run_code_quality_check":        ["LINT_VIOLATION", "STYLE_ERROR", "COMPLEXITY_HIGH"],
    "execute_test_suite":            ["TEST_ASSERTION_FAILED", "TEST_TIMEOUT",
                                     "MISSING_COVERAGE", "IMPORT_ERROR"],
    "scan_security_vulnerabilities": ["CVE_FOUND_CRITICAL", "CVE_SCAN_TIMEOUT",
                                      "REGISTRY_UNAVAILABLE", "SCAN_RATE_LIMITED"],
    "trigger_deployment":            ["DEPLOY_AUTH_FAILED", "CLUSTER_UNAVAILABLE",
                                      "ROLLBACK_TRIGGERED"],
}

OUTPUT_OK = {
    "run_code_quality_check":        " No violations found. Score: {s}/100",
    "execute_test_suite":            " {p} passed, 0 failed. Coverage: {c}%",
    "scan_security_vulnerabilities": " 0 critical CVEs. {l} low-severity advisories.",
    "trigger_deployment":            " Deployed to {e}. Build #{b} live.",
}


def _simulate_step(tool: str, step_no: int) -> dict:
    cfg = FAILURE_CONFIG[tool]
    r   = random.random()

    # Determine status
    if r < cfg["timeout_rate"]:
        status, error_code = "timeout", f"{tool.upper()[:6]}_TIMEOUT"
        latency = BASE_LATENCY[tool] * random.uniform(4, 8)
        output  = f" Tool timed out after {latency:.0f}ms"
    elif r < cfg["timeout_rate"] + cfg["fail_rate"]:
        status     = "failed"
        error_code = random.choice(ERROR_CODES[tool])
        latency    = BASE_LATENCY[tool] * random.uniform(0.4, 1.1)
        output     = f"✗ {error_code}: {tool} failed on step {step_no}"
    else:
        status, error_code = "ok", None
        slow = random.random() < cfg["slow_rate"]
        mult = cfg["slow_multiplier"] if slow else random.uniform(0.85, 1.15)
        latency = BASE_LATENCY[tool] * mult + random.gauss(0, 30)
        latency = max(latency, 50)
        s = random.randint(78, 100)
        p = random.randint(120, 400)
        output = OUTPUT_OK[tool].format(
            s=s, p=p, c=random.randint(72, 98),
            l=random.randint(0, 5), e=random.choice(ENVS),
            b=random.randint(1000, 9999)
        )

    return {"step_no": step_no, "tool": tool, "status": status,
            "latency_ms": round(latency, 1), "output": output,
            "error_code": error_code}


def generate_logs(n_runs: int = 50) -> List[dict]:
    logs   = []
    t_start = datetime(2025, 6, 10, 6, 0, 0)

    for i in range(n_runs):
        triggered_at = t_start + timedelta(minutes=i * random.randint(25, 75))
        repo         = random.choice(REPOS)
        pipeline_id  = f"pipe-{i+1:03d}"
        steps        = []
        run_status   = "success"

        for step_no, tool in enumerate(TOOL_NAMES, 1):
            step = _simulate_step(tool, step_no)
            steps.append(step)
            # Pipeline aborts on failure or timeout
            if step["status"] in ("failed", "timeout"):
                run_status = step["status"]
                # Mark remaining steps as skipped
                for remaining_tool in TOOL_NAMES[step_no:]:
                    steps.append({"step_no": step_no + 1, "tool": remaining_tool,
                                  "status": "skipped", "latency_ms": 0,
                                  "output": "Skipped — upstream step failed",
                                  "error_code": None})
                break

        total_ms = sum(s["latency_ms"] for s in steps)
        logs.append({
            "run_id":       str(uuid.uuid4())[:6],
            "pipeline_id":  pipeline_id,
            "repo":         repo,
            "triggered_at": triggered_at.isoformat(),
            "status":       run_status,
            "total_ms":     round(total_ms, 1),
            "steps":        steps,
        })

    return logs


# Generate the dataset
LOGS = generate_logs(50)
print(f"Log dataset generated: {len(LOGS)} pipeline runs")
print(f"Date range: {LOGS[0]['triggered_at'][:16]}  →  {LOGS[-1]['triggered_at'][:16]}")

Planning Agent setup - complete the TODOs


### Task 1 — Define the `LogEvent` dataclass
A `LogEvent` captures a single moment in the agent's execution.

**Every log event must record:**
| Field | Type | Description |
|-------|------|-------------|
| `event_id` | `str` | A short unique ID (first 6 chars of a UUID) |
| `event_type` | `str` | One of: `"reasoning"`, `"tool_call"`, `"tool_result"`, `"final_result"` |
| `timestamp` | `str` | UTC timestamp as ISO string |
| `content` | `Any` | The actual data being logged (text, dict, etc.) |
| `latency_ms` | `float` | How long this step took in milliseconds (default `0.0`) |
| `metadata` | `Dict` | Any extra key/value pairs (default empty dict) |

---

In [ ]:
# ── Preview one log entry so you can see the structure before starting ────────
sample = LOGS[4]
print(json.dumps(sample, indent=2))

## 3. Task 1 — Explore the Log Structure and Compute Basic Statistics

Before diagnosing any problems, you need to understand what the data looks like at a high level.

### What to compute:
1. **Overall counts** — total runs, and how many were `success`, `failed`, or `timeout`
2. **Overall success rate** — as a percentage
3. **Response time stats** — min, max, mean, and median `total_ms` across all runs
4. **Per-repo breakdown** — for each repo, count total runs and successful runs
5. **Flatten the steps** — build a flat list of all individual step records (you'll need this in every later task)

### Expected output format:
=== BASIC STATISTICS ===

Total runs   : 50

success    : XX  (XX.X%)

failed     : XX  (XX.X%)

timeout    : XX  (XX.X%)

Response time (ms):

Min    : XXXX

Max    : XXXX

Mean   : XXXX

Median : XXXX

Per-repo run counts:

backend-api       : total=XX  success=XX


**Hint:** Use `Counter` from `collections` for counting statuses.  
**Hint:** To flatten steps, loop over `LOGS` and then over each run's `"steps"` list.  
**Hint:** Add `"run_id"`, `"repo"`, and `"triggered_at"` from the parent run into each flattened step dict — you'll need them later for joining.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 1 — Explore logs and compute basic statistics
# ─────────────────────────────────────────────────────────────────────────────

# ── 1a. Count run statuses ────────────────────────────────────────────────────
# TODO: Use Counter to count how many runs have each status
#  YOUR CODE HERE
status_counts = ...


# ── 1b. Compute success rate ──────────────────────────────────────────────────
# TODO: Calculate success_rate as a float between 0 and 1
#  YOUR CODE HERE
success_rate = ...


# ── 1c. Compute response time statistics ──────────────────────────────────────
# TODO: Extract total_ms from all runs, then compute min, max, mean, median
#  YOUR CODE HERE
all_ms      = ...
rt_min      = ...
rt_max      = ...
rt_mean     = ...
rt_median   = ...


# ── 1d. Per-repo breakdown ────────────────────────────────────────────────────
# TODO: Build a dict { repo_name: {"total": int, "success": int} }
#  YOUR CODE HERE
repo_stats = ...


# ── 1e. Flatten all steps into a single list ──────────────────────────────────
# TODO: Create all_steps — a flat list of step dicts.
#       Each dict should contain all original step fields PLUS:
#         "run_id", "repo", "triggered_at", "run_status" from the parent run.
#  YOUR CODE HERE
all_steps = ...


# ── Print the report ──────────────────────────────────────────────────────────
# TODO: Print the formatted statistics report shown in the instructions above
#  YOUR CODE HERE

## 4. Task 2 — Identify Failure Patterns

Now that you have the basic landscape, it is time to dig into **what is failing and why**.

### What to compute:
1. **Per-tool failure stats** — for each of the 4 tools, compute:
   - Total calls (excluding `"skipped"` steps)
   - Number of `"failed"` calls
   - Number of `"timeout"` calls
   - Combined failure rate `(failed + timeout) / total_calls`
2. **Error code frequency** — count how many times each `error_code` appears across all steps (ignore `None`)
3. **Failure cascade analysis** — for every run that failed, identify *which step number* caused the abort. Count how often each step position (1, 2, 3, 4) is the failure point.
4. **Co-occurrence check** — find any runs where **two or more steps** failed or timed out (these point to systemic problems, not one-off errors)
5. **Print a ranked tool failure report** sorted by failure rate descending

### Expected output includes:
- A ranked table showing tool name, total calls, failures, timeouts, and failure rate
- Top 5 most frequent error codes
- Which step position breaks most pipelines

 **Hint:** For step 1, filter `all_steps` to exclude entries where `step["status"] == "skipped"`.  
 **Hint:** For the cascade analysis, look at the **first step with a non-ok, non-skipped status** in each failed run's steps list.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 2 — Identify failure patterns
# ─────────────────────────────────────────────────────────────────────────────

# ── 2a. Per-tool failure stats ────────────────────────────────────────────────
# TODO: Build tool_stats dict:
#   { tool_name: {"total": int, "failed": int, "timeout": int, "fail_rate": float} }
#   Exclude skipped steps from counts.
#  YOUR CODE HERE
tool_stats = ...


# ── 2b. Error code frequency ──────────────────────────────────────────────────
# TODO: Count occurrences of each error_code across all_steps (skip None values)
# Store result as a Counter or dict sorted by frequency descending
#  YOUR CODE HERE
error_code_counts = ...


# ── 2c. Failure cascade analysis ──────────────────────────────────────────────
# TODO: For each failed/timeout run, find which step_no caused the abort.
#       Build a Counter: { step_no: count_of_times_this_step_caused_abort }
#  YOUR CODE HERE
abort_at_step = ...


# ── 2d. Co-occurrence — runs with multiple failures ───────────────────────────
# TODO: Find runs where 2+ steps have status "failed" or "timeout"
#       Store as a list of run_ids
#  YOUR CODE HERE
multi_failure_runs = ...


# ── Print the failure pattern report ─────────────────────────────────────────
# TODO: Print a ranked table of tool failure rates, top error codes,
#       abort-step distribution, and multi-failure run count.
#  YOUR CODE HERE

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 2 — Visualisation (fill in the data — chart code is provided)
# ─────────────────────────────────────────────────────────────────────────────
# TODO: Fill in the three variables below using your results from Task 2.
#       The chart code will run automatically once they are defined.

# tool_names_ordered  — list of tool names sorted by failure rate descending
#  YOUR CODE HERE
tool_names_ordered = ...   # e.g. ["scan_security_vulnerabilities", ...]

# failure_rates  — list of failure rates (0.0–1.0) in the same order
#  YOUR CODE HERE
failure_rates = ...        # e.g. [0.40, 0.16, ...]

# top_errors  — list of (error_code, count) tuples, top 6 by count
#  YOUR CODE HERE
top_errors = ...           # e.g. [("CVE_FOUND_CRITICAL", 8), ...]

# ── Chart (do not modify below this line) ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Failure Pattern Analysis", fontsize=14, fontweight="bold")

SHORT = {"run_code_quality_check":       "CodeQuality",
         "execute_test_suite":           "TestSuite",
         "scan_security_vulnerabilities":"SecScan",
         "trigger_deployment":           "Deployment"}

short_names  = [SHORT.get(n, n) for n in tool_names_ordered]
bar_colours  = ["#E74C3C" if r > 0.20 else "#F39C12" if r > 0.10 else "#27AE60"
                for r in failure_rates]

ax = axes[0]
bars = ax.bar(short_names, [r * 100 for r in failure_rates],
              color=bar_colours, edgecolor="white", linewidth=0.8)
for b, v in zip(bars, failure_rates):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5,
            f"{v*100:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.axhline(10, color="#E74C3C", linestyle="--", linewidth=1.2, label="10% threshold")
ax.set_title("Tool Failure Rates", fontweight="bold")
ax.set_ylabel("Failure Rate (%)")
ax.set_ylim(0, max([r*100 for r in failure_rates]) * 1.35)
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

ax2 = axes[1]
err_labels = [e[0].replace("_", "\n") for e in top_errors]
err_counts = [e[1] for e in top_errors]
err_cols   = ["#E74C3C"] * len(err_labels)
bars2 = ax2.barh(err_labels, err_counts, color=err_cols, edgecolor="white",
                 linewidth=0.8, alpha=0.85)
for b, v in zip(bars2, err_counts):
    ax2.text(v + 0.1, b.get_y() + b.get_height() / 2,
             str(v), va="center", fontsize=9, fontweight="bold")
ax2.set_title("Top Error Codes by Frequency", fontweight="bold")
ax2.set_xlabel("Occurrences")
ax2.invert_yaxis()
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

## 5.  Task 3 — Detect Performance Anomalies

Failures are not the only problem. Some tools may be **degraded without fully failing** — they complete successfully but take much longer than normal. These "slow" runs are often the early warning sign before an outage.

### What to compute:
1. **Baseline latency per tool** — using only `"ok"` steps, compute the mean and standard deviation of `latency_ms` for each tool
2. **Outlier detection** — flag any step as a **latency anomaly** if its `latency_ms` exceeds `mean + 2 × std` for that tool
3. **Anomaly rate per tool** — what percentage of ok-status calls were anomalously slow?
4. **Slowest runs overall** — identify the top 5 runs by `total_ms` and print their pipeline ID, repo, status, and total duration
5. **Hourly pattern** — group runs by the hour of day they were triggered and compute mean `total_ms` per hour to check for time-of-day degradation

### Expected output includes:
- Table showing baseline mean/std and anomaly count/rate per tool
- Top 5 slowest runs
- Hourly mean response time (printed as a table or chart)

 **Hint:** For step 1, filter `all_steps` to `status == "ok"` before computing per-tool stats.  
 **Hint:** For step 2, loop through `all_steps` again and compare each step's latency against the per-tool threshold you computed in step 1.  
 **Hint:** For the hourly pattern, extract the hour with `datetime.fromisoformat(run["triggered_at"]).hour`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 3 — Detect performance anomalies
# ─────────────────────────────────────────────────────────────────────────────

# ── 3a. Baseline latency per tool (ok-status steps only) ─────────────────────
# TODO: For each tool, compute mean and std of latency_ms from ok-status steps only.
#       Store as: tool_baseline = { tool_name: {"mean": float, "std": float} }
#  YOUR CODE HERE
tool_baseline = ...


# ── 3b. Flag latency anomalies ────────────────────────────────────────────────
# TODO: Loop through all_steps (ok status only).
#       A step is an anomaly if latency_ms > mean + 2 * std for its tool.
#       Build: anomalies = list of step dicts that are anomalous
#  YOUR CODE HERE
anomalies = ...


# ── 3c. Anomaly rate per tool ─────────────────────────────────────────────────
# TODO: For each tool, compute:
#   anomaly_rate = count of anomalous ok-steps / total ok-steps
#   Store as: anomaly_rates = { tool_name: float }
#  YOUR CODE HERE
anomaly_rates = ...


# ── 3d. Top 5 slowest runs overall ───────────────────────────────────────────
# TODO: Sort LOGS by total_ms descending and take the top 5.
#       Print pipeline_id, repo, status, and total_ms for each.
#  YOUR CODE HERE
slowest_runs = ...


# ── 3e. Hourly pattern ────────────────────────────────────────────────────────
# TODO: Group LOGS by hour of day (0–23). For each hour that has runs,
#       compute mean total_ms. Store as: hourly_ms = { hour_int: mean_ms }
#  YOUR CODE HERE
hourly_ms = ...


# ── Print the anomaly report ──────────────────────────────────────────────────
# TODO: Print tool_baseline stats, anomaly counts/rates, and top 5 slowest runs
#  YOUR CODE HERE

## 6. Task 4 — Build a Failure Pattern Classifier and Timeline

Now that you know **which** tools fail and **how** slowly they degrade, you need to **classify the failure patterns** so the on-call engineer knows exactly what they're dealing with.

### Define 4 failure pattern categories:

| Pattern | Rule | Severity |
|---------|------|---------|
| `CRITICAL_SECURITY` | Any step with error_code containing `"CVE"` or `"SCAN"` |  Critical |
| `TEST_REGRESSION` | Any step with error_code containing `"TEST"` or `"COVERAGE"` |  High |
| `INFRA_INSTABILITY` | Any step with status `"timeout"` OR error_code containing `"UNAVAILABLE"` or `"CLUSTER"` |  Medium |
| `CODE_QUALITY_DRIFT` | Any step with error_code containing `"LINT"` or `"STYLE"` or `"COMPLEXITY"` |  Low |

### What to build:
1. **`classify_failure(step: dict) -> Optional[str]`** — returns the pattern name for a failed/timeout step, or `None` if it doesn't match any pattern
2. **`classify_run(run: dict) -> List[str]`** — returns a list of all unique patterns found in that run's steps
3. **Pattern frequency table** — count how many runs were tagged with each pattern
4. **Failure timeline** — build a list of `(triggered_at, pattern, repo)` tuples for all classified failures, sorted by time
5. **Print the top 3 most recent failures** for the most common pattern

 **Hint:** For `classify_failure`, check the conditions in priority order (CRITICAL first) and return the first match.  
 **Hint:** For the timeline, a single run can produce multiple timeline entries if it has multiple distinct patterns.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 4 — Failure pattern classifier and timeline
# ─────────────────────────────────────────────────────────────────────────────

# ── 4a. classify_failure() ───────────────────────────────────────────────────
# TODO: Implement classify_failure(step) → Optional[str]
#       Check conditions in priority order: CRITICAL_SECURITY first, then
#       TEST_REGRESSION, INFRA_INSTABILITY, CODE_QUALITY_DRIFT.
#       Return None if the step has no error_code and status is "ok" or "skipped".
#  YOUR CODE HERE
def classify_failure(step: dict):
    pass


# ── 4b. classify_run() ───────────────────────────────────────────────────────
# TODO: Implement classify_run(run) → List[str]
#       Loop through run["steps"], call classify_failure on each,
#       collect unique non-None results.
#  YOUR CODE HERE
def classify_run(run: dict):
    pass


# ── 4c. Pattern frequency table ──────────────────────────────────────────────
# TODO: Apply classify_run() to every run in LOGS.
#       Count how many runs are tagged with each pattern.
#       Store as: pattern_counts = Counter({ pattern_name: int })
#  YOUR CODE HERE
pattern_counts = ...


# ── 4d. Failure timeline ─────────────────────────────────────────────────────
# TODO: Build failure_timeline — a list of dicts, one per (run × pattern) pair:
#   { "triggered_at": str, "pattern": str, "repo": str, "run_id": str }
#   Sort the list by "triggered_at" ascending.
#  YOUR CODE HERE
failure_timeline = ...


# ── 4e. Top 3 most recent failures for the dominant pattern ──────────────────
# TODO: Find the most common pattern from pattern_counts.
#       Print the 3 most recent timeline entries for that pattern.
#  YOUR CODE HERE

## 7. 🛠️ Task 5 — Write a Structured System Health Report

The on-call engineer needs a **written report** they can read in under two minutes.  
Using all the data you computed in Tasks 1–4, implement `print_health_report()`.

### The report must contain exactly these 5 sections:

```
╔══════════════════════════════════════════════════════════════════╗
║        SYSTEM HEALTH REPORT — CI/CD Pipeline Agent              ║
║        Period: 2025-06-10 06:00  →  [last run triggered_at]     ║
╚══════════════════════════════════════════════════════════════════╝

[1] EXECUTIVE SUMMARY
  Overall status : 🔴 CRITICAL  /  🟠 DEGRADED  /  🟢 HEALTHY
  Success rate   : XX.X%
  Total runs     : 50  (success: XX | failed: XX | timeout: XX)

[2] TOP ISSUES (ranked by severity)
  🔴 CRITICAL_SECURITY   : XX runs affected — Security scanner returning CVE findings or timing out
  🟠 TEST_REGRESSION     : XX runs affected — Test suite failures — assertions or coverage drops
  🟡 INFRA_INSTABILITY   : XX runs affected — Timeouts and service unavailability across tools
  🔵 CODE_QUALITY_DRIFT  : XX runs affected — Linting violations and style/complexity warnings

[3] WORST PERFORMING COMPONENT
  Tool           : [name of tool with highest failure rate]
  Failure rate   : XX.X%  (XX failed, XX timeout out of XX calls)
  Top error code : [most frequent error_code for that tool]
  Recommendation : [1–2 sentence action derived from the dominant failure pattern]

[4] PERFORMANCE ANOMALIES
  Latency anomalies detected : XX steps across XX tool(s)
  Slowest run  : [pipeline_id] — [total_ms]ms ([repo], status=[status])
  Hourly peak  : Hour HH:00 UTC — avg XXXXms

[5] RECOMMENDED ACTIONS
  1. [Action derived from the #1 severity pattern with most runs affected]
  2. [Action derived from the #2 severity pattern with runs affected]
  3. [Action derived from the worst performing tool / anomaly data]
```

---

### Rules for each section:

**[1] Executive Summary — Overall Status logic:**

| Condition | Status |
|-----------|--------|
| `success_rate < 0.70` | 🔴 CRITICAL |
| `success_rate < 0.90` | 🟠 DEGRADED |
| `success_rate >= 0.90` | 🟢 HEALTHY |

---

**[2] Top Issues — Severity order is fixed:**  
Always print in this order regardless of count: `CRITICAL_SECURITY` → `TEST_REGRESSION` → `INFRA_INSTABILITY` → `CODE_QUALITY_DRIFT`.  
Mark the most frequently occurring pattern with `◄ DOMINANT` at the end of its line.

---

**[3] Worst Performing Component:**  
- Find the tool with the highest `fail_rate` from your `tool_stats` dict  
- Find its most common `error_code` by scanning `all_steps` filtered to that tool  
- The recommendation should come from `PATTERN_ACTION` — a dict you define that maps each pattern name to a 1–2 sentence action string

---

**[4] Performance Anomalies:**  
- `total_anomalies` = `len(anomalies)` from Task 3  
- `tools_with_anomalies` = number of distinct tool names that appear in your `anomalies` list  
- `slowest_run` = first entry from your `slowest_runs` list (Task 3)  
- `peak_hour` = the hour key with the highest value in `hourly_ms` (Task 3)

---

**[5] Recommended Actions:**  
Iterate over the 4 patterns in severity order. For each pattern that has `count > 0`, add its action string to a pool. Print the first 3 items from that pool numbered 1, 2, 3.  
This means the actions are **data-driven** — if `CODE_QUALITY_DRIFT` has 0 runs, its action won't appear.

---

###  Hints:

- Define a `PATTERN_DESC` dict and a `PATTERN_ACTION` dict at the top of your function mapping each of the 4 pattern names to a description string and an action string respectively. This keeps the report logic clean.
- Use `f"╔{'═' * 66}╗"` style formatting for the box border — or simply hardcode the border width to 66 characters.
- For the `worst_tool_top_err`, filter `error_code_counts` to only codes that appear in steps belonging to that tool: `[ec for ec, _ in error_code_counts.most_common() if any(ec == s.get("error_code") for s in all_steps if s["tool"] == worst_tool)]` — take the first result.
- `dominant` pattern = `max(pattern_counts, key=pattern_counts.get)` — compute this before building the report so you can reference it in sections [2] and [3].

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 5 — Structured system health report
# ─────────────────────────────────────────────────────────────────────────────

def print_health_report():
    \"\"\"
    Print a structured 5-section system health report using all data
    computed in Tasks 1–4. See the markdown cell above for the required format.
    \"\"\"

    # TODO: Implement the full health report
    # Use variables from Tasks 1–4:
    #   status_counts, success_rate, all_ms, tool_stats, error_code_counts,
    #   pattern_counts, anomalies, slowest_runs, hourly_ms, failure_timeline
    #  YOUR CODE HERE
    pass


# Run the report
print_health_report()

## 8. Validation — Run This Cell to Check Your Work

In [ ]:
# ── Validation checks — do not modify ────────────────────────────────────────
print("Running validation checks...\n")
errors = []

# Task 1 checks
try:
    assert isinstance(status_counts, Counter), "status_counts should be a Counter"
    assert sum(status_counts.values()) == 50,  "status_counts should total 50 runs"
    assert 0 < success_rate < 1,               "success_rate should be between 0 and 1"
    assert len(all_steps) > 0,                 "all_steps should not be empty"
    assert "run_id" in all_steps[0],           "all_steps entries should include run_id"
    print(f"  Task 1: Basic stats correct ({len(all_steps)} steps flattened)")
except AssertionError as e:
    errors.append(f"  Task 1: {e}")

# Task 2 checks
try:
    assert len(tool_stats) == 4,               "tool_stats should have 4 tools"
    assert all("fail_rate" in v for v in tool_stats.values()), \
        "each tool_stats entry needs a 'fail_rate' key"
    assert isinstance(error_code_counts, (Counter, dict)), \
        "error_code_counts should be a Counter or dict"
    assert len(error_code_counts) > 0,         "error_code_counts should not be empty"
    print(f"  Task 2: Failure patterns computed ({len(error_code_counts)} error codes)")
except AssertionError as e:
    errors.append(f"  Task 2: {e}")

# Task 3 checks
try:
    assert len(tool_baseline) == 4,            "tool_baseline should have 4 tools"
    assert all("mean" in v and "std" in v for v in tool_baseline.values()), \
        "tool_baseline entries need 'mean' and 'std'"
    assert isinstance(anomalies, list),         "anomalies should be a list"
    assert len(hourly_ms) > 0,                 "hourly_ms should not be empty"
    print(f"  Task 3: Anomalies detected ({len(anomalies)} latency anomalies)")
except AssertionError as e:
    errors.append(f"  Task 3: {e}")

# Task 4 checks
try:
    assert callable(classify_failure),          "classify_failure should be a function"
    assert callable(classify_run),              "classify_run should be a function"
    test_step = {"status": "failed", "error_code": "CVE_FOUND_CRITICAL",
                 "tool": "scan_security_vulnerabilities", "latency_ms": 900}
    assert classify_failure(test_step) == "CRITICAL_SECURITY", \
        "classify_failure should return CRITICAL_SECURITY for CVE error code"
    assert isinstance(pattern_counts, (Counter, dict)), \
        "pattern_counts should be a Counter or dict"
    assert isinstance(failure_timeline, list),  "failure_timeline should be a list"
    print(f"  Task 4: Classifier works, {len(failure_timeline)} timeline events")
except AssertionError as e:
    errors.append(f"  Task 4: {e}")

# Task 5 checks
try:
    assert callable(print_health_report),       "print_health_report should be callable"
    print("  Task 5: print_health_report() is defined")
except AssertionError as e:
    errors.append(f"  Task 5: {e}")

print()
if errors:
    for err in errors:
        print(" ", err)
else:
    print("All checks passed! Your log analysis is complete.")